# ARTERY Feedback Tutorial

This notebook gives a compact end-to-end tutorial for the reproduced ARTERY-style low-latency quantum feedback flow. It combines the previous split notebooks into one file and keeps the result figures next to the notebook under `results/`.

## 1. Tutorial Flow

```text
S21 readout traces
  -> IQ demodulation
  -> state classification
  -> segmented trajectory features
  -> confidence / BHT-style early prediction
  -> feedback branch and waveform selection
```

The ARTERY idea is to avoid waiting for a full software round trip. Hardware analyzes the readout trajectory while samples arrive and emits feedback when the confidence or predictor state is sufficient.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../software/s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in 3-artery/tutorial/ or 3-artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

## 2. Load S21 Readout Data

The input data contains repeated readout shots for the `|0>` and `|1>` states. Each shot is a time sequence of I/Q samples.

In [ ]:
read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
print('MAT keys:', sorted(read_data.keys()))
print('data shape:', read_data['data'].shape)
print('|0> I/Q:', read_zero_i.shape, read_zero_q.shape)
print('|1> I/Q:', read_one_i.shape, read_one_q.shape)
print('measurement fidelities:', read_data.get('measure_fids'))

### Mean Q Traces

![Mean Q trace](results/original_notebook/cell_07_output_01.png)

![Mean state Q traces](results/original_notebook/cell_08_output_01.png)

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(np.mean(read_zero_q, axis=0), label='mean Q |0>')
plt.plot(np.mean(read_one_q, axis=0), label='mean Q |1>')
plt.xlabel('sample index')
plt.ylabel('Q amplitude')
plt.legend()
plt.tight_layout()

## 3. IQ Demodulation

The software demodulation is the reference model for the hardware NCO + mixer + accumulator path:

```text
I_demod = sum(I * cos(wt) + Q * sin(wt))
Q_demod = sum(Q * cos(wt) - I * sin(wt))
```

In [ ]:
idx1, idx2 = 1, 2000
omega = OMEGAS[2]
result_zero = demod_part(omega, read_zero_i[idx1:idx2], read_zero_q[idx1:idx2])
result_one = demod_part(omega, read_one_i[idx1:idx2], read_one_q[idx1:idx2])

plt.figure(figsize=(5, 5))
plt.scatter(result_zero[:, 0], result_zero[:, 1], s=8, alpha=0.5, label='|0>')
plt.scatter(result_one[:, 0], result_one[:, 1], s=8, alpha=0.5, label='|1>')
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.legend()
plt.tight_layout()

### Demodulated IQ Feature View

![Demodulated IQ features](results/original_notebook/cell_11_output_00.png)

## 4. State Classification

The tutorial uses KMeans offline to estimate the two state clusters. Hardware should not run KMeans online; it should use fixed centers or thresholds exported from software.

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

features = np.vstack([result_zero, result_one])
true_labels = np.array([0] * len(result_zero) + [1] * len(result_one))
kmeans = KMeans(n_clusters=2, random_state=0, n_init='auto').fit(features)
labels = kmeans.labels_
acc = max(metrics.accuracy_score(true_labels, labels), metrics.accuracy_score(true_labels, 1 - labels))
print('cluster centers:')
print(kmeans.cluster_centers_)
print('best label accuracy:', acc)

plt.figure(figsize=(5, 5))
plt.scatter(features[:, 0], features[:, 1], c=labels, cmap='viridis', s=8, alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=120, c='red')
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.tight_layout()

### KMeans State Clustering

![KMeans state clustering](results/original_notebook/cell_13_output_01.png)

## 5. Segmented Demodulation and Early Prediction

ARTERY can make a feedback decision before the full readout ends. The stream is accumulated over a configured window, and confidence is checked at segment boundaries.

In [ ]:
window_start = 850
window_len = 200
idx1, idx2 = 0, 2000

seg_zero = demod_part(OMEGAS[0], read_zero_i[idx1:idx2, window_start:window_start + window_len], read_zero_q[idx1:idx2, window_start:window_start + window_len])
seg_one = demod_part(OMEGAS[0], read_one_i[idx1:idx2, window_start:window_start + window_len], read_one_q[idx1:idx2, window_start:window_start + window_len])
seg_features = np.vstack([seg_zero, seg_one])
seg_labels = np.array([0] * len(seg_zero) + [1] * len(seg_one))
seg_pred = KMeans(n_clusters=2, random_state=0, n_init='auto').fit(seg_features)
seg_acc = max(metrics.accuracy_score(seg_labels, seg_pred.labels_), metrics.accuracy_score(seg_labels, 1 - seg_pred.labels_))
print('window_start:', window_start, 'window_len:', window_len, 'best segmented accuracy:', seg_acc)

### Segmented Results

![Segmented demodulation](results/original_notebook/cell_15_output_02.png)

![Prediction accuracy vs window](results/original_notebook/cell_18_output_00.png)

### Hardware Decision Rule

```text
if P(state1) >= threshold_hi: emit branch1 feedback
elif P(state1) <= threshold_lo: emit branch0 feedback
else: continue accumulating the next segment
```

If no threshold is crossed, the implementation can force a final classification at the maximum readout window.

## 6. Trajectory Analyzer and Window Search

The trajectory analyzer converts accumulated I/Q features into a sequence of partial readout points. Window search finds a latency/accuracy tradeoff for feedback.

In [ ]:
window_base, window_cnt, window_len = 850, 6, 300
shots, base_shot = 2, 122

def trajectory(read_i, read_q, shot, omega):
    points = []
    for step in range(window_cnt):
        end = window_base + (step + 1) * window_len
        feat = demod_part(omega, read_i[shot:shot + 1, window_base:end], read_q[shot:shot + 1, window_base:end])
        points.append(feat[0])
    return np.array(points)

plt.figure(figsize=(6, 5))
for shot in range(shots):
    z = trajectory(read_zero_i, read_zero_q, base_shot + shot, OMEGAS[0])
    o = trajectory(read_one_i, read_one_q, base_shot + shot, OMEGAS[0])
    plt.plot(z[:, 0], z[:, 1], 'o-.', color='skyblue', label='|0>' if shot == 0 else None)
    plt.plot(o[:, 0], o[:, 1], 'o-.', color='orange', label='|1>' if shot == 0 else None)
plt.xlabel('integrated I')
plt.ylabel('integrated Q')
plt.legend()
plt.tight_layout()

### Trajectory Figures

![Demodulation trajectory](results/original_notebook/cell_21_output_00.png)

![Trajectory comparison](results/original_notebook/cell_22_output_02.png)

## 7. BHT-Style Predictor

The full ARTERY direction includes predictor structures such as a trajectory analyzer, branch history table, and Bayesian-style confidence estimator. The compact tutorial code below builds a small BHT-like table from recent trajectory bits.

In [ ]:
window_start = 850
step = 100
num_steps = 16
omega = OMEGAS[0]

def trajectory_bits(read_i, read_q, center_zero, center_one, shots):
    bits = []
    for k in range(num_steps):
        length = (k + 1) * step
        feat = demod_part(omega, read_i[:shots, window_start:window_start + length], read_q[:shots, window_start:window_start + length])
        d0 = np.linalg.norm(feat - center_zero, axis=1)
        d1 = np.linalg.norm(feat - center_one, axis=1)
        p1 = d0 / (d0 + d1 + 1e-12)
        bits.append((p1 >= 0.5).astype(np.uint8))
    return np.stack(bits, axis=1)

train_shots = 1000
full_len = num_steps * step
train_zero = demod_part(omega, read_zero_i[:train_shots, window_start:window_start + full_len], read_zero_q[:train_shots, window_start:window_start + full_len])
train_one = demod_part(omega, read_one_i[:train_shots, window_start:window_start + full_len], read_one_q[:train_shots, window_start:window_start + full_len])
center_zero, center_one = train_zero.mean(axis=0), train_one.mean(axis=0)

bits0 = trajectory_bits(read_zero_i, read_zero_q, center_zero, center_one, train_shots)
bits1 = trajectory_bits(read_one_i, read_one_q, center_zero, center_one, train_shots)
train_bits = np.vstack([bits0, bits1])
train_labels = np.array([0] * train_shots + [1] * train_shots)

history_len = 8
bht = {}
for bits, label in zip(train_bits, train_labels):
    for end in range(history_len, bits.shape[0] + 1):
        key = ''.join(map(str, bits[end - history_len:end]))
        zeros, ones = bht.get(key, [0, 0])
        zeros += int(label == 0)
        ones += int(label == 1)
        bht[key] = [zeros, ones]

bht_prob = {key: ones / (zeros + ones) for key, (zeros, ones) in bht.items()}
print('BHT entries:', len(bht_prob))
print('example entries:', list(bht_prob.items())[:5])

## 8. Hardware Interface and GUI

The full Vivado project is intentionally not included in this repository. The tutorial keeps the packet format, datapath description, and Verilog boundary under `hw/interface/`.

In [ ]:
for path in sorted(Path('../hw/interface').glob('*')):
    print(path)

print('
--- feedback_datapath.md ---')
print(Path('../hw/interface/feedback_datapath.md').read_text()[:1500])

### GUI Result

![ARTERY GUI result](results/artery_gui_current.png)